In [1]:
from google.colab import files
import pandas as pd
import io

uploaded = files.upload()

# Use the first uploaded CSV file
filename = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[filename]))

# Clean column names
df.columns = df.columns.str.strip()

required_columns = [
    "ID",
    "section_header",
    "section_text",
    "dialogue"
]

missing = set(required_columns) - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

# Remove completely empty records and normalize text fields
df = df.dropna(how="all", subset=required_columns).copy()

for col in required_columns:
    df[col] = df[col].fillna("").astype(str).str.strip()

print(f"Loaded {len(df)} source records.")
display(df.head())

Saving MTS-Dialog-TrainingSet.csv to MTS-Dialog-TrainingSet.csv
Loaded 1201 source records.


,ID,section_header,section_text,dialogue
0,0,GENHX,The patient is a 76-year-old white female who ...,Doctor: What brings you back into the clinic t...
1,1,GENHX,The patient is a 25-year-old right-handed Cauc...,Doctor: How're you feeling today? \r\nPatient...
2,2,GENHX,"This is a 22-year-old female, who presented to...","Doctor: Hello, miss. What is the reason for yo..."
3,3,MEDICATIONS,Prescribed medications were Salmeterol inhaler...,Doctor: Are you taking any over the counter me...
4,4,CC,"Burn, right arm.","Doctor: Hi, how are you? \r\nPatient: I burned..."


In [2]:
import os
import re
import json
import time
import hashlib
import pandas as pd
from pathlib import Path

from tqdm.auto import tqdm
from google.colab import userdata
from google import genai
from google.genai import types

# Output paths
OUTPUT_DIR = Path("/content/mts_dialog_processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = OUTPUT_DIR / "dataset.csv"
CORPUS_PATH = OUTPUT_DIR / "corpus.jsonl"

# Setup client
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
client = genai.Client()
MODEL_NAME = "gemini-3.8-flash"
BATCH_INPUT_FILE = "mts_dialogue_batch_input.jsonl"

# Split configuration
DEV_RATIO = 0.80
RANDOM_SEED = 42

# Question source:
# "dialogue" uses the complete dialogue field as the query.
# Change this if you have a separate extracted question field.
QUESTION_SOURCE = "dialogue"

# Keep False until gold answers and scenario labels are verified.
STRICT_VALIDATION = False

ALLOWED_SCENARIOS = {
    "A_SUFFICIENT",
    "B_MISSING",
    "C_DISTRACTING"
}

ALLOWED_SPLITS = {"dev", "test"}

In [3]:
def make_question_id(source_id: str) -> str:
    """
    Create a deterministic ID from the original source ID.
    """
    source_id = str(source_id).strip()

    if not source_id:
        raise ValueError(
            "Encountered a record with an empty source ID."
        )

    digest = hashlib.sha256(
        source_id.encode("utf-8")
    ).hexdigest()[:16].upper()

    return f"QA_MED_{digest}"


def assign_split(question_id: str) -> str:
    """
    Deterministically assign a record to dev or test.
    """
    digest = hashlib.sha256(
        f"{RANDOM_SEED}:{question_id}".encode("utf-8")
    ).hexdigest()

    fraction = int(digest[:8], 16) / 0xFFFFFFFF

    return "dev" if fraction < DEV_RATIO else "test"


def build_context(section_header: str, section_text: str) -> str:
    """
    Combine section metadata and narrative without
    adding empty or duplicated sections.
    """
    header = clean_text(section_header)
    narrative = clean_text(section_text)

    if header and narrative:
        return f"{header}\n\n{narrative}"

    return header or narrative


def clean_text(text: str) -> str:
    """
    Normalize line endings and excess whitespace.
    """
    if pd.isna(text):
        return ""

    text = str(text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


In [4]:
SYSTEM_PROMPT = """
You are a medical dataset annotation assistant.

Transform the provided doctor-patient dialogue and
clinical context into one structured medical QA record.

Generate exactly three fields:

1. question:
   Generate one clinically meaningful question based
   on the patient's encounter and available information.

2. gold_answer:
   Provide a factual reference answer supported by
   the original dialogue and clinical context.

3. scenario:
   Provide a concise description of the clinical
   encounter, including relevant conditions and purpose.

STRICT RULES:

- Use only information provided in the source.
- Do not invent symptoms, diagnoses, treatments,
  medications, or clinical findings.
- Preserve important negative findings.
- Do not infer missing clinical information.
- Ensure the question is answerable using the source.
- Ensure the gold_answer directly addresses the question.
- Use professional medical English.
- Do not add medical advice unsupported by the source.
- Do not include patient identifiers unnecessarily.

Return valid JSON with exactly these fields:
{
    "question": "...",
    "gold_answer": "...",
    "scenario": "..."
}
"""


In [ ]:
# ==========================================
# PHASE 1: GENERATE BATCH ENTRIES
# ==========================================
batch_entries = []

for _, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Preparing Batch Payloads"
):
    # 1. Generate deterministic question ID
    question_id = make_question_id(row["ID"])

    # 2. Extract original dialogue
    dialogue = clean_text(row[QUESTION_SOURCE])

    # 3. Build clinical context
    context = build_context(
        row["section_header"],
        row["section_text"]
    )
    context = clean_text(context)

    # 4. Validate source data
    if not dialogue:
        raise ValueError(f"Empty dialogue for source ID {row['ID']!r}.")
    if not context:
        raise ValueError(f"Empty context for source ID {row['ID']!r}.")

    # 5. Construct prompt matching your engineering schema
    user_prompt = f"""
DIALOGUE:
{dialogue}

CLINICAL CONTEXT:
{context}

Generate the required medical annotations.
"""

    # 6. Format structurally for the Google Batch Engine using standard API schema
    # Note: We must store structural metadata (dialogue, context) so we can map it back in Phase 2
    metadata_payload = {
        "question_id": question_id,
        "dialogue": dialogue,
        "context": context
    }

    batch_request = {
        "key": question_id, # Unique identifier string
        "request": {
            "contents": [{"parts": [{"text": user_prompt}]}],
            # 1. systemInstruction is a TOP-LEVEL field in the request object (camelCase)
            "systemInstruction": {
                "parts": [{"text": SYSTEM_PROMPT}]
            },
            # 2. generationConfig handles output parameters (camelCase)
            "generationConfig": {
                "temperature": 0.0,
                "responseMimeType": "application/json"
            }
        }
    }


    # Save a tuple containing the API request dictionary and local backup metadata
    batch_entries.append((batch_request, metadata_payload))

# ==========================================
# WRITE FILE & TRANSMIT TO GOOGLE STORAGE
# ==========================================
# Write the API request components to a line-delimited JSON
with open(BATCH_INPUT_FILE, "w") as f:
    for req_dict, _ in batch_entries:
        f.write(json.dumps(req_dict) + "\n")

print(f"\nCompiled {len(batch_entries)} records into local file: {BATCH_INPUT_FILE}")

# Upload payload to Google Files server
uploaded_file = client.files.upload(
    file=BATCH_INPUT_FILE,
    config=types.UploadFileConfig(
        display_name="mts_dialogue_annotations",
        mime_type="jsonl"
    )
)
print(f"Uploaded successfully to URI: {uploaded_file.name}")

# Launch asynchronous processing cluster
batch_job = client.batches.create(
    model=MODEL_NAME,
    src=uploaded_file.name,
    config={"display_name": "mts_dialogue_processing_run"}
)

# Store this text string! You need it if your runtime disconnects
ACTIVE_JOB_ID = batch_job.name
print(f"\n🚀 Batch Job Started!")
print(f"============================================")
print(f"STORE THIS JOB ID: {ACTIVE_JOB_ID}")
print(f"============================================")

Preparing Batch Payloads:   0%|          | 0/1201 [00:00<?, ?it/s]


Compiled 1201 records into local file: mts_dialogue_batch_input.jsonl
Uploaded successfully to URI: files/87zpkpa3kpzc

🚀 Batch Job Started!
STORE THIS JOB ID: batches/js5q46007iirkmwn8ttr5o03damxs9ofdpfn


In [5]:
import json
import re
import os
import time
import pandas as pd
from google.colab import userdata
from google import genai

# ==========================================
# 1. SETUP ENVIRONMENT CONFIGURATION
# ==========================================
# Make sure your GEMINI_API_KEY is available in Colab Secrets (🔑)
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
client = genai.Client()

# IF YOU ARE RECONNECTING: Paste your stored job ID string here.
# IF THIS IS A CONTINUOUS FIRST RUN: Leave this as None; it will read from Phase 1.
RECONNECT_JOB_ID = "batches/js5q46007iirkmwn8ttr5o03damxs9ofdpfn"

# Resolve which Job identifier string token to target
if RECONNECT_JOB_ID:
    TARGET_JOB_ID = RECONNECT_JOB_ID
    print(f"🔄 Reconnecting Mode Enabled! Targeting explicit Job ID: {TARGET_JOB_ID}")
else:
    try:
        TARGET_JOB_ID = ACTIVE_JOB_ID
        print(f"🚀 Continuous Mode Enabled! Targeting live session Job ID: {TARGET_JOB_ID}")
    except NameError:
        raise NameError("Missing targeting reference! If you disconnected, paste the string into RECONNECT_JOB_ID.")

# Assert vital runtime dependencies are loaded in session memory cache
if 'df' not in globals():
    raise NameError("Critical source DataFrame 'df' not found in workspace memory! Re-run your loading cells first.")
if 'RANDOM_SEED' not in globals():
    print("⚠️ WARNING: 'RANDOM_SEED' variable undefined. Initializing default value: RANDOM_SEED = 42")
    RANDOM_SEED = 42
if 'QUESTION_SOURCE' not in globals():
    QUESTION_SOURCE = "dialogue"

# ==========================================
# 2. MONITOR AND AWAIT PROCESSING END
# ==========================================
print(f"\nPolling status for batch job cluster: {TARGET_JOB_ID}")
while True:
    job_status = client.batches.get(name=TARGET_JOB_ID)
    state = str(job_status.state).upper() # Normalize string format variations
    print(f"Current Processing Status: {state}")

    if "SUCCEEDED" in state or "COMPLETED" in state:
        print("🎉 Batch processing successfully completed on Google Servers!")
        break
    elif "FAILED" in state or "CANCELED" in state:
        raise RuntimeError(f"Google Batch cluster execution aborted with status: {state}")

    time.sleep(30) # Poll engine status every 30 seconds

# ==========================================
# 3. CONVERT METADATA OBJECTS & DOWNLOAD FILE
# ==========================================
# Extract the target file locator from the deep nested structure dynamically
if hasattr(job_status, "model_dump"):
    job_dict = job_status.model_dump()
else:
    job_dict = getattr(job_status, "__dict__", {})

google_output_uri = None
def find_file_uri(data_structure):
    if isinstance(data_structure, dict):
        for k, v in data_structure.items():
            if isinstance(v, str) and (v.startswith("files/") or "batch" in v) and not v.startswith("batches/"):
                return v
            res = find_file_uri(v)
            if res: return res
    elif isinstance(data_structure, list):
        for item in data_structure:
            res = find_file_uri(item)
            if res: return res
    return None

google_output_uri = find_file_uri(job_dict)

if not google_output_uri or google_output_uri.startswith("batches/"):
    job_token = TARGET_JOB_ID.split("/")[-1]
    google_output_uri = f"files/batch-{job_token}"

print(f"Targeting File URI reference: {google_output_uri}")

print("Fetching remote cloud file configuration object...")
file_object = client.files.get(name=google_output_uri)

print("Downloading file byte string stream into local runtime workspace...")
# Resolves SDK bytes download format variations cleanly
downloaded_raw_bytes = client.files.download(file=file_object)
file_content_str = downloaded_raw_bytes.decode("utf-8")
lines = file_content_str.splitlines()
print(f"Successfully unpacked {len(lines)} lines from memory payload buffer.")

# ==========================================
# 4. RECONSTRUCT METADATA LOOKUP INDEX
# ==========================================
print("\nRebuilding core dataset lookup mapping structures...")
source_lookup = {}
for _, row in df.iterrows():
    qid = make_question_id(row["ID"])
    source_lookup[qid] = {
        "dialogue": clean_text(row[QUESTION_SOURCE]),
        "context": clean_text(build_context(row["section_header"], row["section_text"]))
    }

# ==========================================
# 5. PARSE GENERATED CONTENT AND MERGE RECORDS
# ==========================================
records = []
for idx, line in enumerate(lines):
    if not line.strip():
        continue

    try:
        line_data = json.loads(line)
        question_id = line_data.get("key")

        source_meta = source_lookup.get(question_id)
        if not source_meta:
            print(f"Warning: Discovered unmatched request identifier {question_id} in output stream.")
            continue

        # Parse nested candidate structures returned by Google's response block
        response_payload = line_data.get("response", line_data)
        if "candidates" in response_payload:
            candidates = response_payload["candidates"]
            if isinstance(candidates, list) and len(candidates) > 0:
                raw_response_text = candidates[0]["content"]["parts"][0]["text"]
            elif isinstance(candidates, dict):
                raw_response_text = candidates["content"]["parts"][0]["text"]
        else:
            raw_response_text = response_payload.get("output", "")

        # Clean string from any potential markdown layout artifacts
        if "```" in raw_response_text:
            raw_response_text = re.sub(r"```json\s*|\s*```", "", raw_response_text)

        annotation = json.loads(raw_response_text.strip())

        # Enforce validation requirements
        required_fields = ["question", "gold_answer", "scenario"]
        for field in required_fields:
            if field not in annotation:
                raise ValueError(f"Missing field: {field}")
            annotation[field] = clean_text(annotation[field])
            if not annotation[field]:
                raise ValueError(f"Empty field: {field}")

        records.append({
            "question_id": question_id,
            "question": annotation["question"],
            "gold_answer": annotation["gold_answer"],
            "scenario": annotation["scenario"],
            "split": assign_split(question_id),
            "context": source_meta["context"],
            "dialogue": source_meta["dialogue"]
        })

    except Exception as e:
        print(f"Skipping corrupt parse signature on record index {idx}: {e}")

# ==========================================
# 6. CONVERT TO DATAFRAME & RUN CHECKS
# ==========================================
processed = pd.DataFrame(records)

if processed["question_id"].duplicated().any():
    duplicates = processed.loc[processed["question_id"].duplicated(keep=False), "question_id"].tolist()
    raise ValueError(f"Duplicate question IDs found: {duplicates}")

required_df_fields = ["question_id", "question", "gold_answer", "scenario", "split", "context"]
assert processed[required_df_fields].notnull().all().all()
assert (
    processed[required_df_fields]
    .astype(str)
    .apply(lambda col: col.str.strip().ne(""))
    .all()
    .all()
)

# Export output to CSV instantly to keep records completely safe!
processed.to_csv("final_annotated_dataset.csv", index=False)

print("\n==========================================")
print("Records processed cleanly:", len(processed))
print("\nSplit distribution:")
print(processed["split"].value_counts())
print("\nSaved permanently as: final_annotated_dataset.csv")
print("==========================================")

display(processed.head(5))

🔄 Reconnecting Mode Enabled! Targeting explicit Job ID: batches/js5q46007iirkmwn8ttr5o03damxs9ofdpfn

Polling status for batch job cluster: batches/js5q46007iirkmwn8ttr5o03damxs9ofdpfn
Current Processing Status: JOBSTATE.JOB_STATE_SUCCEEDED
🎉 Batch processing successfully completed on Google Servers!
Targeting File URI reference: files/batch-js5q46007iirkmwn8ttr5o03damxs9ofdpfn
Fetching remote cloud file configuration object...
Successfully unpacked 1201 lines from memory payload buffer.

Rebuilding core dataset lookup mapping structures...

Records processed cleanly: 1201

Split distribution:
split
dev     953
test    248
Name: count, dtype: int64

Saved permanently as: final_annotated_dataset.csv


,question_id,question,gold_answer,scenario,split,context,dialogue
0,QA_MED_5FECEB66FFC86F38,"What is the patient's past medical history, an...","The patient has a history of hypertension, ost...",A 76-year-old female presents for a medication...,dev,GENHX\n\nThe patient is a 76-year-old white fe...,Doctor: What brings you back into the clinic t...
1,QA_MED_6B86B273FF34FCE1,What presenting symptoms did the patient repor...,The patient presented with a sudden-onset seve...,A 25-year-old female presents to the emergency...,dev,GENHX\n\nThe patient is a 25-year-old right-ha...,Doctor: How're you feeling today? \nPatient: T...
2,QA_MED_D4735E3A265E16EE,What presenting symptoms and duration does the...,The patient reports noticing warts (condylomas...,A 22-year-old female presents to an outpatient...,test,"GENHX\n\nThis is a 22-year-old female, who pre...","Doctor: Hello, miss. What is the reason for yo..."
3,QA_MED_4E07408562BEDB8B,What medications is the patient currently pres...,The patient is prescribed a Salmeterol inhaler...,A clinical encounter reviewing and reconciling...,dev,MEDICATIONS\n\nPrescribed medications were Sal...,Doctor: Are you taking any over the counter me...
4,QA_MED_4B227777D4DD1FC6,"What injury does the patient present with, and...",The patient presents with a burn injury isolat...,A patient consults a physician regarding an ac...,dev,"CC\n\nBurn, right arm.","Doctor: Hi, how are you? \nPatient: I burned m..."


In [6]:
# ==========================================
# 1. FILE CONFIGURATION PATHS & CONSTRAINTS
# ==========================================
# CUSTOM OUTPUT STRUCTURE
OUTPUT_DIR = Path("/content/mts_dialog_processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Define path configurations using your specific directory hooks
DATASET_PATH = OUTPUT_DIR / "dataset.csv"
CORPUS_PATH = OUTPUT_DIR / "corpus.jsonl"

ALLOWED_SPLITS = {"train", "validation", "test", "dev"}

if 'processed' not in globals() or processed.empty:
    raise NameError("The 'processed' DataFrame was not found or is empty. Run your Phase 2 cell first!")

# ==========================================
# 2. GENERATE SPLIT DATA STRUCTURES
# ==========================================
print("Extracting evaluation matrix parameters...")
dataset = processed[
    ["question_id", "gold_answer", "scenario", "split"]
].copy()

print("Extracting clinical corpus parameters...")
corpus = processed[
    ["question_id", "question", "context"]
].rename(
    columns={"question_id": "id"}
)

# ==========================================
# 3. DATA INTEGRITY ASSURANCE AND CHECKS
# ==========================================
detected_splits = set(dataset["split"].dropna().unique())
if not detected_splits.issubset(ALLOWED_SPLITS):
    invalid_splits = detected_splits - ALLOWED_SPLITS
    raise ValueError(f"Dataframe contains unmapped split signatures: {invalid_splits}")

# ==========================================
# 4. EXPORT TO SPECIFIC DIRECTORY PATHS
# ==========================================
# Save the evaluation dataset (converted Path to string)
dataset.to_csv(
    str(DATASET_PATH),
    index=False,
    encoding="utf-8"
)

# Save the text corpus (converted Path to string)
with open(str(CORPUS_PATH), "w", encoding="utf-8") as f:
    for record in corpus.to_dict(orient="records"):
        f.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

# ==========================================
# 5. RENDER DIAGNOSTIC REVIEWS
# ==========================================
print("\n==========================================")
print("Files exported successfully to target directory:")
print(f" -> Evaluation CSV Matrix: {DATASET_PATH.resolve()}")
print(f" -> Clinical JSONL Corpus: {CORPUS_PATH.resolve()}")
print("==========================================")

print("\n📊 Dataset Evaluation Matrix Preview:")
display(dataset.head(3))

print("\n🧬 First Corpus Record Signature Check:")
print(json.dumps(
    corpus.iloc[0].to_dict(),
    indent=2,
    ensure_ascii=False
))


Extracting evaluation matrix parameters...
Extracting clinical corpus parameters...

Files exported successfully to target directory:
 -> Evaluation CSV Matrix: /content/mts_dialog_processed/dataset.csv
 -> Clinical JSONL Corpus: /content/mts_dialog_processed/corpus.jsonl

📊 Dataset Evaluation Matrix Preview:


,question_id,gold_answer,scenario,split
0,QA_MED_5FECEB66FFC86F38,"The patient has a history of hypertension, ost...",A 76-year-old female presents for a medication...,dev
1,QA_MED_6B86B273FF34FCE1,The patient presented with a sudden-onset seve...,A 25-year-old female presents to the emergency...,dev
2,QA_MED_D4735E3A265E16EE,The patient reports noticing warts (condylomas...,A 22-year-old female presents to an outpatient...,test



🧬 First Corpus Record Signature Check:
{
  "id": "QA_MED_5FECEB66FFC86F38",
  "question": "What is the patient's past medical history, and what acute symptoms were assessed and denied during the visit?",
  "context": "GENHX\n\nThe patient is a 76-year-old white female who presents to the clinic today originally for hypertension and a med check. She has a history of hypertension, osteoarthritis, osteoporosis, hypothyroidism, allergic rhinitis and kidney stones. Since her last visit she has been followed by Dr. Kumar. Those issues are stable. She has had no fever or chills, cough, congestion, nausea, vomiting, chest pain, chest pressure."
}


In [7]:
# Check exact headers
expected_dataset_columns = [
    "question_id",
    "gold_answer",
    "scenario",
    "split"
]

expected_corpus_columns = [
    "id",
    "question",
    "context"
]

assert list(dataset.columns) == expected_dataset_columns
assert list(corpus.columns) == expected_corpus_columns

# Check ID correspondence
assert dataset["question_id"].tolist() == corpus["id"].tolist()

# Reload saved files to validate serialization
dataset_check = pd.read_csv(
    DATASET_PATH,
    dtype=str,
    keep_default_na=False
)

corpus_check = []

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        try:
            record = json.loads(line)
        except json.JSONDecodeError as exc:
            raise ValueError(
                f"Invalid JSON on line {line_number}"
            ) from exc

        if set(record.keys()) != {"id", "question", "context"}:
            raise ValueError(
                f"Unexpected fields on line {line_number}"
            )

        corpus_check.append(record)

assert len(dataset_check) == len(corpus_check)

assert (
    dataset_check["question_id"].tolist()
    == [r["id"] for r in corpus_check]
)

print("All structural checks passed.")
print("Rows in dataset.csv:", len(dataset_check))
print("Rows in corpus.jsonl:", len(corpus_check))

All structural checks passed.
Rows in dataset.csv: 1201
Rows in corpus.jsonl: 1201


In [8]:
from google.colab import files

files.download(str(DATASET_PATH))
files.download(str(CORPUS_PATH))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>